The following code, written with the help of ChatGPT, queries data about World War 1 medals from Europeana, and saves 3 .csv files, one being a sample of 100 entries, another being the entire data queried, and the last one being a filtered version with only relevant columns included. Note that OUTPUT_FOLDER should be changed before running the code.

In [1]:
# High-precision WWI medals dataset + column-filtered export from Europeana

import requests
import pandas as pd
import time
import os
import re

# =============================
# CONFIGURATION
ROWS_PER_PAGE = 100
OUTPUT_FOLDER = r"C:\Users\Admin\openscholarship\practicedata"

FULL_DATASET_FILENAME = "wwi_medals_high_precision.csv"
FILTERED_DATASET_FILENAME = "wwi_medals_filtered_columns.csv"
SAMPLE_FILENAME = "wwi_medals_sample_100.csv"
# =============================

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# -----------------------------
# Step 1: Broad Europeana query
# -----------------------------
print("Searching Europeana (broad query)...\n")

base_url = "https://api.europeana.eu/record/v2/search.json"

query = (
    '(medal OR medals) AND ('
    '"World War I" OR "First World War" OR "1914-1918" OR '
    '"Erster Weltkrieg" OR "Première Guerre mondiale"'
    ')'
)

params = {
    "query": query,
    "wskey": "apidemo",
    "rows": ROWS_PER_PAGE,
    "cursor": "*",
    "profile": "rich",
    "media": "true"
}

all_items = []
page = 1

while True:
    print(f"Fetching page {page}…")
    response = requests.get(base_url, params=params)

    if response.status_code != 200:
        print("Error:", response.status_code, response.text)
        break

    data = response.json()
    items = data.get("items", [])
    all_items.extend(items)

    next_cursor = data.get("nextCursor")
    if not next_cursor:
        print("No more pages.")
        break

    params["cursor"] = next_cursor
    page += 1
    time.sleep(0.2)

print("\nTotal items retrieved:", len(all_items))

# -----------------------------
# Step 2: Normalize dataset
# -----------------------------
df = pd.json_normalize(all_items)
print("\nDataset shape (raw):", df.shape)

# -----------------------------
# Step 3: Build combined text
# -----------------------------
text_fields = [col for col in df.columns if any(k in col.lower() for k in ["title", "subject", "type", "description"])]

df["combined_text"] = (
    df[text_fields]
    .astype(str)
    .agg(" ".join, axis=1)
    .str.lower()
)

# -----------------------------
# Step 4: HIGH-PRECISION FILTER
# -----------------------------
df_medals = df[
    df["combined_text"].str.contains(r"\bmedal(s)?\b", regex=True, na=False)
]

EXCLUDE_TERMS = [
    "coin", "banknote", "stamp",
    "poster", "photograph", "painting",
    "drawing", "print", "map"
]

exclude_pattern = "|".join(EXCLUDE_TERMS)

df_medals = df_medals[
    ~df_medals["combined_text"].str.contains(exclude_pattern, regex=True, na=False)
]

if "edmPreview" in df_medals.columns:
    df_medals = df_medals[df_medals["edmPreview"].notna()]

print("\nDataset shape (filtered, high precision):", df_medals.shape)

# -----------------------------
# Step 5: Save full dataset
# -----------------------------
full_output_path = os.path.join(OUTPUT_FOLDER, FULL_DATASET_FILENAME)
df_medals.to_csv(full_output_path, index=False)
print("\nSaved full dataset to:", full_output_path)

# -----------------------------
# Step 6: Save filtered columns dataset
# -----------------------------
SELECTED_COLUMNS = [
    "dcSubjectLangAware.en",
    "dctermsSpatial",
    "edmTimespanLabelLangAware.en",
    "edmConceptPrefLabelLangAware.en",
    "year",
    "type",
    "title",
    "provider",
    "dataProvider",
    "country",
    "completeness",
    "dcLanguageLangAware.def"
]

# Keep only existing columns (important for Europeana data)
available_columns = [col for col in SELECTED_COLUMNS if col in df_medals.columns]

df_filtered_columns = df_medals[available_columns].copy()

filtered_output_path = os.path.join(OUTPUT_FOLDER, FILTERED_DATASET_FILENAME)
df_filtered_columns.to_csv(filtered_output_path, index=False)

print("\nSaved filtered column dataset to:", filtered_output_path)
print("Columns included:", available_columns)

# -----------------------------
# Step 7: Save 100-item sample
# -----------------------------
sample_size = min(100, len(df_medals))
df_sample = df_medals.sample(sample_size, random_state=42)

sample_output_path = os.path.join(OUTPUT_FOLDER, SAMPLE_FILENAME)
df_sample.to_csv(sample_output_path, index=False)

print("Saved 100-item sample to:", sample_output_path)

# -----------------------------
# Step 8: Quick inspection
# -----------------------------
print("\nSample titles:")
print(df_sample["title"].head(10))

Searching Europeana (broad query)...

Fetching page 1…
Fetching page 2…
Fetching page 3…
Fetching page 4…
Fetching page 5…
Fetching page 6…
Fetching page 7…
Fetching page 8…
Fetching page 9…
Fetching page 10…
Fetching page 11…
Fetching page 12…
Fetching page 13…
Fetching page 14…
Fetching page 15…
Fetching page 16…
Fetching page 17…
Fetching page 18…
Fetching page 19…
Fetching page 20…
Fetching page 21…
Fetching page 22…
Fetching page 23…
Fetching page 24…
Fetching page 25…
Fetching page 26…
Fetching page 27…
Fetching page 28…
Fetching page 29…
Fetching page 30…
Fetching page 31…
Fetching page 32…
Fetching page 33…
Fetching page 34…
Fetching page 35…
Fetching page 36…
Fetching page 37…
Fetching page 38…
Fetching page 39…
Fetching page 40…
Fetching page 41…
Fetching page 42…
Fetching page 43…
Fetching page 44…
Fetching page 45…
Fetching page 46…
Fetching page 47…
Fetching page 48…
Fetching page 49…
No more pages.

Total items retrieved: 4782

Dataset shape (raw): (4782, 300)


C:\Users\Admin\AppData\Local\Temp\ipykernel_15316\1611473648.py:92: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["combined_text"].str.contains(r"\bmedal(s)?\b", regex=True, na=False)



Dataset shape (filtered, high precision): (2073, 301)

Saved full dataset to: C:\Users\Admin\openscholarship\practicedata\wwi_medals_high_precision.csv

Saved filtered column dataset to: C:\Users\Admin\openscholarship\practicedata\wwi_medals_filtered_columns.csv
Columns included: ['dcSubjectLangAware.en', 'dctermsSpatial', 'edmTimespanLabelLangAware.en', 'edmConceptPrefLabelLangAware.en', 'year', 'type', 'title', 'provider', 'dataProvider', 'country', 'completeness', 'dcLanguageLangAware.def']
Saved 100-item sample to: C:\Users\Admin\openscholarship\practicedata\wwi_medals_sample_100.csv

Sample titles:
2323    [Général von Linsingen : Médailles et décorati...
2140    [01 Certificat de médaille militaire de Charle...
3205    [Croix de guerre étoile de bronze de René Jacq...
1036    [Βασίλης Ηλία Τσιάρτας (Macedonian Mule Corps ...
1626    [Familieboekje trouwboekje, Marriage certificate]
2114                                   [Le soldat décoré]
1060    [The Comptons, A small family co

The current plan is to look further into ways of cleaning and better querying the data within the limitations and challenges of working with data gathered from Europeana, which is not uniform.

The code below, written with the help of ChatGPT, compiles statistical data on the metadata queried in the previous code. The columns of "provider", "dataProvider", and "country" are included.

In [2]:
# Metadata statistics for provider / dataProvider / country
# Creates a text report with value frequencies

import pandas as pd
import os

# =====================================
# CONFIGURATION
INPUT_CSV = r"C:\Users\Admin\openscholarship\practicedata\wwi_medals_filtered_columns.csv"

OUTPUT_FOLDER = r"C:\Users\Admin\openscholarship\practicedata"

OUTPUT_TEXT_FILE = "metadata_statistics_report.txt"
# =====================================

# Load dataset
df = pd.read_csv(INPUT_CSV)

# Columns to analyze
TARGET_COLUMNS = [
    "provider",
    "dataProvider",
    "country",
    "dcLanguageLangAware.def"
]

# Output path
output_path = os.path.join(OUTPUT_FOLDER, OUTPUT_TEXT_FILE)

# =====================================
# Generate report
# =====================================

with open(output_path, "w", encoding="utf-8") as f:

    f.write("EUROPEANA WWI MEDALS METADATA REPORT\n")
    f.write("=" * 60 + "\n\n")

    for col in TARGET_COLUMNS:

        f.write(f"\nCOLUMN: {col}\n")
        f.write("-" * 60 + "\n")

        # Check if column exists
        if col not in df.columns:
            f.write("Column not found in dataset.\n")
            continue

        # Value counts including missing values
        counts = df[col].fillna("MISSING_VALUE").value_counts()

        # Summary stats
        f.write(f"Total unique values: {counts.shape[0]}\n")
        f.write(f"Total rows: {len(df)}\n\n")

        f.write("Value frequencies:\n\n")

        # Write each value + frequency
        for value, freq in counts.items():
            f.write(f"{value} --> {freq}\n")

        f.write("\n")

print(f"Metadata statistics report saved to:\n{output_path}")

Metadata statistics report saved to:
C:\Users\Admin\openscholarship\practicedata\metadata_statistics_report.txt


Some issues and problems about Europeana's data not being uniform are visible with the output of the metadata analysis.

In order to try to infer the country of origin of entries from the default language column, a seperate code cell for the metadata statistical analysis of entries with Europe in their country column is added below that will only analyse the column "dcLanguageLangAware.def".

In [3]:
# Metadata statistics (filtered to "Europe" in country column)

import pandas as pd
import os

# =============================
# CONFIGURATION
INPUT_CSV = r"C:\Users\Admin\openscholarship\practicedata\wwi_medals_filtered_columns.csv"
OUTPUT_FOLDER = r"C:\Users\Admin\openscholarship\practicedata"
OUTPUT_TEXT_FILE = "metadata_statistics_report_europe_only.txt"
# =============================

# Load dataset
df = pd.read_csv(INPUT_CSV)

# -----------------------------
# Step 1: Filter for Europe only
# -----------------------------
if "country" not in df.columns:
    raise ValueError("Column 'country' not found in dataset")

df_europe = df[
    df["country"].astype(str).str.contains("Europe", case=False, na=False)
].copy()

print(f"Rows after Europe filter: {len(df_europe)}")

# -----------------------------
# Step 2: Target columns
# -----------------------------
TARGET_COLUMNS = ["dcLanguageLangAware.def"]

output_path = os.path.join(OUTPUT_FOLDER, OUTPUT_TEXT_FILE)

# -----------------------------
# Step 3: Generate report
# -----------------------------
with open(output_path, "w", encoding="utf-8") as f:

    f.write("EUROPEANA WWI MEDALS METADATA REPORT (EUROPE FILTERED)\n")
    f.write("=" * 70 + "\n\n")

    f.write(f"Total rows after filtering: {len(df_europe)}\n\n")

    for col in TARGET_COLUMNS:

        f.write(f"\nCOLUMN: {col}\n")
        f.write("-" * 70 + "\n")

        if col not in df_europe.columns:
            f.write("Column not found in dataset.\n")
            continue

        # Clean values
        series = df_europe[col].fillna("MISSING_VALUE")

        counts = series.value_counts()

        f.write(f"Unique values: {counts.shape[0]}\n")
        f.write("Value frequencies:\n\n")

        for value, freq in counts.items():
            f.write(f"{value} --> {freq}\n")

        f.write("\n")

print(f"\nReport saved to:\n{output_path}")

Rows after Europe filter: 1512

Report saved to:
C:\Users\Admin\openscholarship\practicedata\metadata_statistics_report_europe_only.txt


The current plan is to try to infer the country of origin from the metadata that was gathered, which is not uniform.